# 01. CSV設計とYouTubeメタデータ取得

## 目的

YouTube Data APIからジャンル別プレイリスト内の動画メタデータを取得し、分析・分類用のCSVを作成する。

今回のPoCでは、1本の動画をCSVの1行として扱う。
自分で作成したジャンル別プレイリストを正解ラベルとして利用する。

## データ作成方針

* 1行 = 1動画
* プレイリストごとにジャンルラベルを付ける
* APIから取得できる情報は raw CSV に広めに保存する
* 分析・学習時には、目的に応じて使う列を選ぶ
* モデル入力には使わない列でも、重複確認・元動画確認・取得元管理・再現性確保のために保存する

## 使用するプレイリスト

| プレイリスト                      | ラベル              | 用途               |
| --------------------------- | ---------------- | ---------------- |
| YTClassify_music            | music            | 学習用              |
| YTClassify_game             | game             | 学習用              |
| YTClassify_study            | study            | 学習用              |
| YTClassify_cooking          | cooking          | 学習用              |
| YTClassify_other_candidates | other_candidates | 学習には使わず、分類結果の観察用 |

## raw CSV設計

| 列名                     | 内容              | 用途                 |
| ---------------------- | --------------- | ------------------ |
| source_playlist_id     | 取得元プレイリストID     | 取得元管理              |
| source_playlist_name   | 取得元プレイリスト名      | 取得元管理              |
| genre                  | 正解ラベル           | 学習に使う              |
| use_for_training       | 学習対象に含めるか       | 学習用データと観察用データの切り分け |
| video_id               | YouTube動画ID     | 重複削除・元動画確認         |
| url                    | YouTube動画URL    | 誤分類例の確認            |
| is_available           | 動画詳細を取得できたか     | 削除済み・非公開動画の確認      |
| title                  | 動画タイトル          | 初期モデルで使う           |
| description            | 動画概要欄           | 初期モデルで使う           |
| channel_id             | チャンネルID         | チャンネル偏り確認          |
| channel_title          | チャンネル名          | 追加特徴量候補            |
| tags                   | 動画タグ            | 追加特徴量候補            |
| category_id            | YouTube上のカテゴリID | 追加特徴量候補            |
| duration               | 動画時間            | EDA・追加特徴量候補        |
| published_at           | 投稿日時            | EDA・データ偏り確認        |
| view_count             | 再生回数            | EDA・追加特徴量候補        |
| like_count             | 高評価数            | EDA・追加特徴量候補        |
| comment_count          | コメント数           | EDA・追加特徴量候補        |
| default_language       | デフォルト言語         | データ確認              |
| default_audio_language | 音声言語            | データ確認              |
| fetched_at             | API取得日時         | 再現性確保・統計情報の取得時点管理  |

## 分類モデルの方針

最初は説明しやすさを優先し、タイトルと概要欄をTF-IDFで特徴量化し、ロジスティック回帰で分類する。

正解ラベルには `genre` を使う。
ただし、`other_candidates` は学習には含めず、学習済みモデルの分類結果を観察するために使う。

## ベースライン

まずは、タイトルと概要欄を単純に結合したテキストをベースラインとする。

```python
text = title + " " + description
```

このベースラインにより、タイトルと概要欄を同じ重みで扱った場合の分類性能を確認する。

## タイトル重み付けの比較

YouTube動画では、概要欄にSNSリンク、定型文、広告文、ハッシュタグなどのノイズが含まれやすい。
一方で、タイトルには動画内容やジャンルを直接表す語が含まれやすいと考えられる。

そのため、タイトルを複数回連結することで、タイトル由来の単語の重みを相対的に高くした特徴量も比較する。

```python
# title_weight = 1
text = title + " " + description

# title_weight = 2
text = title + " " + title + " " + description

# title_weight = 3
text = title + " " + title + " " + title + " " + description
```

実装では、`title_weight` をパラメータとして変えられるようにする。

```python
text = (title + " ") * title_weight + description
```

## 評価方針

各 `title_weight` について、同じ学習データ・テストデータ分割でモデルを学習し、以下を比較する。

* accuracy
* ジャンル別 precision / recall / f1-score
* 混同行列
* 誤分類例

精度だけでなく、タイトル重みを上げることでどの誤分類が減ったか、逆にどの誤分類が増えたかも確認する。

## 今後の拡張

タイトルを複数回連結する方法は実装が簡単で説明しやすい一方、やや力技でもある。

今後は、`title` と `description` を別々にTF-IDF化し、`title` 側の特徴量に明示的な重みを与える方法も検討する。

例えば、`ColumnTransformer` を使うことで、以下のように `title` と `description` を別々の特徴量として扱える。

```python
transformer_weights = {
    "title_tfidf": 2.0,
    "description_tfidf": 1.0,
}
```

今回はまず、`title_weight` を変えた単純な比較実験を行い、タイトル重視が有効かどうかを確認する。

## other_candidates の扱い

`other_candidates` は最初の学習には含めない。

理由は、`other` はジャンルとしての一貫性が低く、教師あり分類の1カテゴリとして学習させるとノイズになりやすいため。

今回は、学習済みモデルに `other_candidates` の動画を入力し、どのジャンルに寄るか、分類確率が低くなるかを観察する用途にする。


In [44]:
# ジャンル別プレイリスト設定

PLAYLISTS = {
    "music": {
        "playlist_id": "PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY",
        "playlist_name": "YTClassify_music",
        "use_for_training": True,
    },
    "game": {
        "playlist_id": "PLpssoRoNCKKx2eeuKDmVjkxq_nL9zGkqH",
        "playlist_name": "YTClassify_game",
        "use_for_training": True,
    },
    "study": {
        "playlist_id": "PLpssoRoNCKKwPbKukYpAgVa0OzlLzaW8N",
        "playlist_name": "YTClassify_study",
        "use_for_training": True,
    },
    "cooking": {
        "playlist_id": "PLpssoRoNCKKwLEJArO9CCHN8e03OjzcZE",
        "playlist_name": "YTClassify_cooking",
        "use_for_training": True,
    },
    "other_candidates": {
        "playlist_id": "PLpssoRoNCKKz-iWS9t9h0pzGF_E8SXEgv",
        "playlist_name": "YTClassify_other_candidates",
        "use_for_training": False,
    },
}

In [45]:
RAW_COLUMNS = [
    "source_playlist_id",
    "source_playlist_name",
    "genre",
    "use_for_training",
    "video_id",
    "url",
    "is_available",
    "title",
    "description",
    "channel_id",
    "channel_title",
    "tags",
    "category_id",
    "duration",
    "published_at",
    "view_count",
    "like_count",
    "comment_count",
    "default_language",
    "default_audio_language",
    "fetched_at",
]

In [46]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
import json

In [47]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

if not YOUTUBE_API_KEY:
    raise ValueError("YOUTUBE_API_KEY が .env に設定されていません")

youtube = build(
    "youtube",
    "v3",
    developerKey=YOUTUBE_API_KEY
)

In [48]:
for genre, config in PLAYLISTS.items():
    if not config["playlist_id"]:
        print(f"{genre}: playlist_id が未設定です")
    else:
        print(f"{genre}: {config['playlist_id']}")

music: PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY
game: PLpssoRoNCKKx2eeuKDmVjkxq_nL9zGkqH
study: PLpssoRoNCKKwPbKukYpAgVa0OzlLzaW8N
cooking: PLpssoRoNCKKwLEJArO9CCHN8e03OjzcZE
other_candidates: PLpssoRoNCKKz-iWS9t9h0pzGF_E8SXEgv


In [49]:
def fetch_playlist_items(youtube, playlist_id, max_pages=None):
    """
    指定したプレイリスト内の動画ID一覧を取得する。
    """
    items = []
    next_page_token = None
    page_count = 0

    while True:
        request = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token,
        )
        response = request.execute()

        for item in response.get("items", []):
            video_id = item["contentDetails"].get("videoId")

            if video_id is None:
                continue

            items.append({
                "video_id": video_id,
            })

        next_page_token = response.get("nextPageToken")
        page_count += 1

        if next_page_token is None:
            break

        if max_pages is not None and page_count >= max_pages:
            break

    return items

In [50]:
def fetch_video_details(youtube, video_ids):
    """
    video_id のリストから、動画の詳細メタデータを取得する。

    YouTube Data API の videos.list に id をカンマ区切りで渡して取得する。
    まとめて取得するため、video_ids は50件ずつに分割する。
    """
    details = []

    if not video_ids:
        return details

    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i + 50]

        request = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(batch_ids),
        )
        response = request.execute()

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            content_details = item.get("contentDetails", {})
            statistics = item.get("statistics", {})

            video_id = item.get("id")

            details.append({
                "video_id": video_id,
                "url": f"https://www.youtube.com/watch?v={video_id}",
                "is_available": True,
                "title": snippet.get("title"),
                "description": snippet.get("description"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "tags": json.dumps(snippet.get("tags", []), ensure_ascii=False),
                "category_id": snippet.get("categoryId"),
                "duration": content_details.get("duration"),
                "published_at": snippet.get("publishedAt"),
                "view_count": statistics.get("viewCount"),
                "like_count": statistics.get("likeCount"),
                "comment_count": statistics.get("commentCount"),
                "default_language": snippet.get("defaultLanguage"),
                "default_audio_language": snippet.get("defaultAudioLanguage"),
            })

    return details

In [51]:
from datetime import datetime, timezone

rows = []
fetched_at = datetime.now(timezone.utc).isoformat()

for genre, config in PLAYLISTS.items():
    playlist_id = config["playlist_id"]
    playlist_name = config["playlist_name"]
    use_for_training = config["use_for_training"]

    if not playlist_id:
        print(f"skip: {genre} は playlist_id が未設定です")
        continue

    print(f"fetch playlist: {playlist_name} ({genre})")

    playlist_items = fetch_playlist_items(youtube, playlist_id)
    video_ids = [item["video_id"] for item in playlist_items]

    video_details = fetch_video_details(youtube, video_ids)
    detail_by_video_id = {
        item["video_id"]: item
        for item in video_details
    }

    for playlist_item in playlist_items:
        video_id = playlist_item["video_id"]
        detail = detail_by_video_id.get(video_id)

        common = {
            "source_playlist_id": playlist_id,
            "source_playlist_name": playlist_name,
            "genre": genre,
            "use_for_training": use_for_training,
            "video_id": video_id,
            "fetched_at": fetched_at,
        }

        if detail is None:
            row = {
                **common,
                "url": f"https://www.youtube.com/watch?v={video_id}",
                "is_available": False,
            }
        else:
            row = {
                **common,
                **detail,
                "fetched_at": fetched_at,
            }

        rows.append(row)

df_raw = pd.DataFrame(rows, columns=RAW_COLUMNS)
df_raw.head()

fetch playlist: YTClassify_music (music)
fetch playlist: YTClassify_game (game)
fetch playlist: YTClassify_study (study)
fetch playlist: YTClassify_cooking (cooking)
fetch playlist: YTClassify_other_candidates (other_candidates)


,source_playlist_id,source_playlist_name,genre,use_for_training,video_id,url,is_available,title,description,channel_id,...,tags,category_id,duration,published_at,view_count,like_count,comment_count,default_language,default_audio_language,fetched_at
0,PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY,YTClassify_music,music,True,ktzbPzmAnnw,https://www.youtube.com/watch?v=ktzbPzmAnnw,True,「胎児の夢」 feat.初音ミク,胎児は母親の胎内にいる間、生物の進化という壮絶な悪夢を見ているっていう説があるらしいです。\...,UCVJns279mOOd92lWVQl_HlA,...,"[""#VOCALOID"", ""#初音ミク""]",10,PT3M33S,2020-03-31T16:35:22Z,425482,18481,176,ja,NaN,2026-06-20T13:05:06.655004+00:00
1,PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY,YTClassify_music,music,True,arlXG7TVdPk,https://www.youtube.com/watch?v=arlXG7TVdPk,True,天使じゃないよ/初音ミク,他人に理想を押し付けるのも、押し付けられるのも等しくグロテスクですよね\n\n歌：初音ミク\...,UCF9BrkN5Z5tqAwuj_s0519g,...,[],22,PT3M36S,2024-06-21T10:00:05Z,140917,7542,134,ja,NaN,2026-06-20T13:05:06.655004+00:00
2,PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY,YTClassify_music,music,True,DNltbDXaKw8,https://www.youtube.com/watch?v=DNltbDXaKw8,True,きおくをみたの/初音ミク(まかろり×いのうつはSA),生きるって、何だろうね。\n\n作詞作曲：まかろり\n(https://twitter.co...,UCB7DcMLTrdCeCENzfZIfKLQ,...,[],22,PT3M45S,2024-09-29T11:00:22Z,133008,5441,176,ja,ja,2026-06-20T13:05:06.655004+00:00
3,PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY,YTClassify_music,music,True,JqvE2fxm80U,https://www.youtube.com/watch?v=JqvE2fxm80U,True,夢憂鬱 / 初音ミク,VocaDuo2024にて制作した曲の初音ミク歌唱verです。\n\n\n₁₀₄は₆番線さん...,UCwqj1NywsBPbd5QZvHf-yEg,...,[],22,PT3M28S,2024-07-08T10:00:06Z,165424,9875,145,ja,en,2026-06-20T13:05:06.655004+00:00
4,PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY,YTClassify_music,music,True,Pmj5CGPaUQ8,https://www.youtube.com/watch?v=Pmj5CGPaUQ8,True,"八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...",▶Music れーしあ\nhttps://x.com/Re_siadesu\n\n▶Illu...,UC90a5GK8XetNIlNF0kPj7yA,...,"[""八月"", ""僕らの犯した間違いの答え合わせ"", ""ボカロ"", ""夏 ボカロ"", ""カゼヒ...",10,PT3M44S,2024-08-23T11:00:27Z,93070,6715,220,ja,NaN,2026-06-20T13:05:06.655004+00:00


In [52]:
duplicate_video_ids = df_raw.loc[
    df_raw["video_id"].duplicated(keep=False),
    ["video_id", "title", "genre", "source_playlist_name", "url"]
].sort_values("video_id")

display(duplicate_video_ids)

,video_id,title,genre,source_playlist_name,url


In [53]:
print("shape:", df_raw.shape)

print("\nジャンル別件数:")
display(df_raw["genre"].value_counts())

print("\n学習対象/非対象:")
display(df_raw["use_for_training"].value_counts())

print("\n取得可否:")
display(df_raw["is_available"].value_counts())

print("\n重複 video_id 数:")
print(df_raw["video_id"].duplicated().sum())

shape: (464, 21)

ジャンル別件数:


genre
music               123
study               111
game                107
cooking             102
other_candidates     21
Name: count, dtype: int64


学習対象/非対象:


use_for_training
True     443
False     21
Name: count, dtype: int64


取得可否:


is_available
True    464
Name: count, dtype: int64


重複 video_id 数:
0


In [54]:
output_path = PROJECT_ROOT / "data/raw/videos_raw.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df_raw.to_csv(output_path, index=False)

print(f"saved: {output_path}")
print(df_raw.shape)

saved: /Users/nishiyamaharuya/Documents/coding/youtube_analyzer/youtube-playlist-classifier-poc/data/raw/videos_raw.csv
(464, 21)
